# 第35章 柱状图（bar / barh）

<!-- module-learning-arc:start -->
> **Matplotlib 模块主线｜第 4 / 12 步：表达趋势与类别比较**
>
> **持续应用背景：** 制作经营周会一页报告：把趋势、比较、分布和异常证据组织成有主次、可直接用于会议的静态页面。
>
> **承接上一阶段：** 折线图（plot）  →  **本章任务：** 柱状图（bar / barh）  →  **下一步：** 散点与气泡图（scatter）
>
> **大作业连接：** 本章练习将成为《经营周会一页报告》的一部分，最终需要从周会问题出发选择互补图形，完成视觉层级、注释审阅与独立导出。
<!-- module-learning-arc:end -->


## 本章场景

前面章节我们学了清洗和汇总数据，可一串数字摆在眼前时，很难一眼看出哪个类别最高、差距多大。


## 本章目标

学完本章，你将能够：

- **理解**：理解「柱状图（bar / barh）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「柱状图（bar / barh）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「柱状图（bar / barh）」并读出其中的结论。


## 适用场景

**背景引入**：前面章节我们学了清洗和汇总数据，可一串数字摆在眼前时，很难一眼看出哪个类别最高、差距多大。柱状图（bar / barh）正是把离散类别的数量、金额或均值“翻译”成柱子长度，让人一眼就能比较不同类别的差异。本章就用前面准备好的 DataFrame，把它变成一张能直接讲结论的柱状图。 打个比方：柱状图就像让每个类别排队比身高——柱子越高，说明这个类别的数量或金额越大；谁高谁矮、差多少，一排出来清清楚楚，特别适合回答「哪个最高、差距多大」这类比较问题。


比较离散类别的数量、金额、均值或组成。


## 数据结构

一列类别和一列指标；分组或堆积图还需要第二个类别维度。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 width 参数从 0.36 改为 0.6，观察柱形间距变化
2. 移除 bar_label 参数，对比有无数值标签的可读性差异
3. 修改 barh 为 bar，将水平柱状图改为垂直布局


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `np.argsort()`、`plt.subplots()`、`ax.barh()`、`ax.bar_label()` | 比较离散类别的数量、金额、均值或组成。 | 数值轴不从零开始夸大差异 |
| 进阶变体 | `np.arange()`、`plt.subplots()`、`ax.bar()`、`ax.set()` | 在基础图表上增加分组、注释、布局或交互 | 类别太多仍使用竖向柱状图 |
| 关键参数 | `width` | 柱宽 | 数值轴不从零开始夸大差异 |
| 关键参数 | `bottom` | 堆积基线 | 类别太多仍使用竖向柱状图 |
| 关键参数 | `barh` | 水平布局 | 用不同颜色装饰同一序列 |
| 关键参数 | `bar_label` | 数值标签 | 数值轴不从零开始夸大差异 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-35 -->
### 数学推导｜比较图中的差值与占比

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜差值回答“多了多少”。** $d_i=x_i-x_{ref}$，保留原单位。

**第 2 步｜比例回答“是基准的几倍”。** $r_i=x_i/x_{ref}$；相对变化是 $r_i-1$。

**第 3 步｜构成占比需要共同分母。** 令 $T=\sum_jx_j$，则 $s_i=x_i/T$，并且

$$
\sum_i s_i=\frac{\sum_i x_i}{T}=1
$$

所以只有互斥且穷尽的类别，才适合解释为整体构成。

**把上面的关系收束为本章计算式：**

$$
d_i=x_i-x_{ref},\qquad s_i=\frac{x_i}{\sum_jx_j}
$$

**符号解释：** $x_{ref}$ 是比较基准，$s_i$ 是类别 $i$ 的总体占比。

**代码对应：** 在绘图前计算差值或占比列，柱长只负责呈现已经定义好的指标。

**使用边界：** 排序、分母范围和是否包含“其他”类别都会改变占比解释。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，因此这里不需要手动 import
#      或 addfont，直接使用即可。

# 1️⃣ 数据导入：读取订单数据（指定列类型降低内存、加快分组）
transactions = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Country": "category"},
)
print(f"数据规模：{len(transactions):,} 行 × {transactions.shape[1]} 列")


In [ ]:
# 2️⃣ 特征工程：构造分析所需字段与聚合结果
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]

# 有效订单：数量与单价均为正（退货/取消行不参与月度统计）
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
completed["month"] = (
    completed["InvoiceDate"].dt.to_period("M").astype("string")
)

# 月度聚合：销售额（元）与订单数
monthly_summary = completed.groupby("month").agg(
    sales=("amount", "sum"), orders=("InvoiceNo", "nunique")
)
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18  # 简化假设：利润约为销售额的 18%

# 区域构成：销售额前 4 国，统计「销售 vs 退货」两部分（单位：万元）
top = completed.groupby("Country")["amount"].sum().nlargest(4).index
rows = transactions[transactions["Country"].isin(top)].copy()
rows["flow"] = np.where(rows["Quantity"] > 0, "销售", "退货")
regional = (
    pd.crosstab(
        rows["Country"].astype(str),
        rows["flow"],
        values=rows["amount"].abs(),
        aggfunc="sum",
    )
    / 10_000
).fillna(0)
regions = regional.index.to_numpy()
online = regional["销售"].to_numpy()
offline = regional["退货"].to_numpy()

# 固定随机种子抽样 2000 条，供分布图使用，保证每次运行结果一致
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"有效订单：{len(completed):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

totals = online + offline
order = np.argsort(totals)
fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.barh(regions[order], totals[order], color="#1a73e8")
ax.bar_label(bars, padding=4)
ax.set(title="各区域总销售额", xlabel="销售额（万元）")
ax.spines[["top", "right", "left"]].set_visible(False)
fig.tight_layout()
plt.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：把 30.4 的横向柱状图改造成“线上销售额”的纵向柱状图。请修改图表参数或数据字段——把 `ax.barh` 换成 `ax.bar`，并把映射到柱子高度的数据从 `totals` 换成 `online`（数据 `regions`、`online`、`offline` 在前面已准备好）。运行后观察坐标轴方向的变化，并用一句话记录：横向与纵向柱状图分别更适合比较哪种结构？


In [ ]:
try:
    # 请在下方填写代码：用 ax.bar(...) 绘制「各区域线上销售额」的纵向柱状图
    # 提示：数据已就绪（regions / online / offline / totals），把下面柱高字段替换为 online 即可
    import matplotlib.pyplot as plt

    # TODO：请在下方完成 —— 练一练：把 30.4 的横向柱状图改造成“线上销售额”的纵向柱状图。请修改图表参数或数据字段——把 ax.barh
    # 换成

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

x = np.arange(len(regions))
width = 0.36
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(x - width / 2, online, width, label="线上", color="#1a73e8")
ax.bar(x + width / 2, offline, width, label="线下", color="#f9ab00")
ax.set(
    title="区域渠道对比",
    ylabel="销售额（万元）",
    xticks=x,
    xticklabels=regions,
)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
plt.show()


## 参数说明

- width：柱宽
- bottom：堆积基线
- barh：水平布局
- bar_label：数值标签


## 结果解读

比较共享零基线上的柱长；堆积图同时读取总量和构成，但中间序列不易精确比较。


## 本章实训：图表只改一个编码

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月", "4月"]
_demo_sales = [120, 150, 138, 190]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 第一个结果怎么读

标题、坐标轴和单位让读者知道图表回答什么问题。没有这些文字，图形即使画出来也不完整。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(months, sales, color="#2563EB")
ax.axhline(
    sum(sales) / len(sales), color="#DC2626", linestyle="--", label="平均值"
)
ax.set_title("月度销售额与平均值")
ax.set_ylabel("销售额（万元）")
ax.legend()
plt.show()


### 第二个结果怎么读

第二个实验把折线改成柱状图，并增加平均线。请说明：哪种图更适合看趋势，哪种图更适合比较单月差异？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：图表能画出但读不懂怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月"]
_demo_sales = [120, 150, 138]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_xlabel("月份")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

图形没有报错不等于结果可用。遇到“看不懂”的图，优先补标题、坐标轴、单位和关键参照线。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 数值轴不从零开始夸大差异
- 类别太多仍使用竖向柱状图
- 用不同颜色装饰同一序列


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

复制最接近的示例，只修改一种视觉编码，并说明阅读任务如何变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：把「总和」换成「线上 vs 退货」分组对比
    # 【目标】从「看总量」进阶到「看构成」：把一根柱子拆成两段颜色。
    import matplotlib.pyplot as plt

    # 起点示例(已可运行)：用 stacked 柱状图，把 online/offline 叠成两段。
    #   - 第一段画 online，第二段以 bottom=online 叠在其上，表示退货；
    #   - 这样每根柱子高度仍是总量，但能看清其中退货占多少。
    x = np.arange(len(regions))
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.bar(x, online, label="线上销售", color="#1a73e8")
    ax.bar(x, offline, bottom=online, label="退货", color="#f9ab00")
    ax.set_xticks(x)
    ax.set_xticklabels(regions)
    ax.set(title="各区域销售构成", ylabel="销售额（万元）")
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()

    # ---- 反思记录：从总量到构成，读图方式改变了什么 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

使用柱形长度比较类别大小，并掌握排序、分组、堆积和水平布局。


### 你已经掌握

- 判断柱状图（bar / barh）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `width` | 柱宽 |
| `bottom` | 堆积基线 |
| `barh` | 水平布局 |
| `bar_label` | 数值标签 |


### 需要注意

- 数值轴不从零开始夸大差异
- 类别太多仍使用竖向柱状图
- 用不同颜色装饰同一序列


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(regions, online, color="#1a73e8")
ax.set(title="各区域线上销售额", ylabel="销售额（万元）")
fig.tight_layout()


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(regions, online, label="线上", color="#1a73e8")
ax.bar(regions, offline, bottom=online, label="线下", color="#f9ab00")
ax.set(title="区域销售渠道构成", ylabel="销售额（万元）")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
